# Chapter 4 — VectorlessRAG 직접 구현

> pymupdf4llm · LangGraph · Agentic Traversal · Bigtable Case
> v2.0 / 2026 · NOWAVE


## 튜토리얼 구성 (4개)

| § | 튜토리얼 | 도구 | 학습 포인트 |
|---|---|---|---|
| 4-1 | PDF → DocumentTree 빌더 | pymupdf4llm | Stack 기반 마크다운 → 트리 파싱 |
| 4-2 | LangGraph Agentic Traversal | langgraph + openai | analyze→descend→retrieve→generate |
| 4-3 | Bigtable 논문 케이스 스터디 | 직접 구현한 시스템 | 4회 LLM 호출 트래버설 추적 |
| 4-4 | 관찰 가능성·튜닝 | retriever.log + MAX_DEPTH | 운영 디버깅 패턴 |

> **참고**: 본 노트북은 Alpha Iterations(Medium)의 `vectorless-rag` 구현을 한국어로 재구성한 것이다. 코드는 약 300줄 규모이며, Chapter 3의 PageIndex 접근법과 직접 비교해본다.

## 0. 환경 준비

```bash
pip install -q openai langgraph pydantic python-dotenv \
   PyMuPDF==1.27.2.2 pymupdf4llm
```

비용은 약 $0.5~$2 (Bigtable 논문 1개 + 질의 1~5개, GPT-4o-mini 기준).

In [15]:
# ─────────────────────────────────────────────────────
# 환경 변수 + 작업 디렉토리 설정
# ─────────────────────────────────────────────────────
import os, json, time, re, urllib.request
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, TypedDict, Annotated
import operator

from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

#os.environ.setdefault("OPENAI_API_KEY", "sk-...")
WORK = Path("./work"); WORK.mkdir(exist_ok=True)
RES  = Path("./work/results"); RES.mkdir(parents=True, exist_ok=True)

client = OpenAI()
MODEL = "gpt-5.4-mini"
print("환경 준비 완료")

환경 준비 완료


### 0.1 Bigtable 논문 PDF 다운로드

Alpha Iterations 글의 케이스 스터디로 Google의 Bigtable 논문(OSDI 2006, 14페이지)을 사용한다. 이 PDF는 헤딩 구조가 명확하고 표·각주가 적절히 섞여 있어 VectorlessRAG의 트리 빌더를 검증하기 좋다.

In [16]:
# ─────────────────────────────────────────────────────
# Bigtable 논문 PDF 다운로드 (Alpha Iterations 케이스)
# ─────────────────────────────────────────────────────
PDF_URL  = ("https://static.googleusercontent.com/media/research.google.com"
            "/en//archive/bigtable-osdi06.pdf")
PDF_PATH = WORK / "bigtable-osdi06.pdf"

if not PDF_PATH.exists():
    print("[↓] Bigtable 논문 다운로드 중 ...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
print(f"PDF: {PDF_PATH} ({PDF_PATH.stat().st_size:,} bytes)")

PDF: work/bigtable-osdi06.pdf (221,214 bytes)


---
## §4-1 PDF → DocumentTree 빌더 (PageIndex 없이)

`pymupdf4llm.to_markdown()`으로 PDF의 헤딩·페이지 정보를 추출하고, Stack 기반 알고리즘으로 마크다운을 계층 트리로 변환한다. 본 §은 본 챕터의 핵심 부분이다.

### 4-1-1. 데이터 모델 — TreeNode와 DocumentTree

PageIndex의 노드와 동일한 schema를 dataclass로 정의한다. 본 모델은 Chapter 5의 Three-Stage Architecture의 `TreeNode` Pydantic 모델로 발전된다.

In [17]:
# ─────────────────────────────────────────────────────
# §4-1-1: 트리 데이터 모델 정의
# ─────────────────────────────────────────────────────
@dataclass
class TreeNode:
    """트리 노드 1개 — PageIndex와 동일한 schema"""
    id: str
    title: str
    level: int                # 0=root, 1=chapter, 2=section, ...
    page_start: int
    page_end: int
    content: str
    children: List["TreeNode"] = field(default_factory=list)
    heading_type: Optional[str] = None  # 'numbered','unnumbered','page'
    summary: str = ""

    def to_dict(self) -> Dict:
        """디버깅·시각화용 dict 변환"""
        return {
            "id": self.id, "title": self.title, "level": self.level,
            "pages": f"{self.page_start}-{self.page_end}",
            "children_count": len(self.children),
            "preview": (self.content[:200] + "...") if len(self.content) > 200 else self.content,
        }

@dataclass
class DocumentTree:
    """전체 트리 컨테이너"""
    document_name: str
    root: TreeNode
    total_pages: int
    source_path: str = ""

    def print_tree(self, node: Optional[TreeNode] = None, indent: int = 0):
        """들여쓰기로 트리 시각화"""
        if node is None:
            node = self.root
            print(f"\n {self.document_name} ({self.total_pages} pages)")
        icon = "📑" if node.level == 0 else "📖" if node.level == 1 else "📄" if node.level == 2 else "📝"
        print("  "*indent + f"{icon} [L{node.level}] {node.title[:60]} (p{node.page_start}-{node.page_end})")
        for c in node.children:
            self.print_tree(c, indent + 1)

print("TreeNode·DocumentTree 정의 완료")

TreeNode·DocumentTree 정의 완료


### 4-1-2. 헤딩 분류 정규식 — 4가지 패턴

학술 논문·기술 보고서의 헤딩 패턴은 다양하다 — 숫자 번호 (`1.`, `2.3.1`), 로마 숫자 (`I.`, `II.`), 알파벳 (`A.`, `B.`), 무번호 (`Abstract`, `Conclusion`). 각 패턴을 분류해두면 후속 처리에서 useful하다.

In [18]:
# ─────────────────────────────────────────────────────
# §4-1-2: 헤딩 분류 정규식 정의
# ─────────────────────────────────────────────────────
HEADING_PATTERNS = {
    "numbered_section": re.compile(r"^(?:\d+\.)+\s+(.+)$"),     # 1., 2.3.1
    "roman_section":    re.compile(r"^(?:[IVX]+)\.?\s+(.+)$", re.IGNORECASE),
    "letter_section":   re.compile(r"^([A-Z])\.\s+(.+)$"),       # A. Methods
    "unnumbered":       re.compile(r"^([A-Z][a-zA-Z\s]{3,50})$"),
}

def classify_heading(title: str) -> str:
    """헤딩 텍스트를 4가지 패턴으로 분류"""
    t = title.strip()
    for kind, pattern in HEADING_PATTERNS.items():
        if pattern.match(t):
            return kind
    return "unknown"

# 분류 시연
print(classify_heading("1. Introduction"))   # numbered_section
print(classify_heading("Abstract"))           # unnumbered
print(classify_heading("II. Related Work"))   # roman_section
print(classify_heading("A. Setup"))           # letter_section

numbered_section
unnumbered
roman_section
letter_section


### 4-1-3. Stack 기반 트리 빌더 — 본 챕터의 핵심 알고리즘

마크다운을 줄 단위로 파싱하며 stack을 사용해 부모-자식 관계를 추적한다.

**핵심 알고리즘**:
- 새 헤딩의 `level`이 stack 최상단의 level보다 **작거나 같으면** stack을 pop (부모로 올라감)
- 새 노드를 현재 stack 최상단의 자식으로 추가
- 새 노드를 stack에 push (다음 자식의 부모가 됨)

In [19]:
# ─────────────────────────────────────────────────────
# §4-1-3: Markdown → DocumentTree (Stack 기반 파싱)
# ─────────────────────────────────────────────────────
import pymupdf4llm

def parse_pdf_to_tree(pdf_path: Path) -> DocumentTree:
    """PDF → 마크다운 → Stack 기반 파싱 → 트리"""
    print(f"Parsing {pdf_path.name} with PyMuPDF4LLM ...")
    t0 = time.time()

    # Step 1: 마크다운 추출 (레이아웃 보존)
    full_md = pymupdf4llm.to_markdown(str(pdf_path))

    # Step 1b: 페이지 청크 (정확한 page range 매칭용)
    page_chunks = pymupdf4llm.to_markdown(
        str(pdf_path), page_chunks=True,
        write_images=False, embed_images=False
    )
    page_contents = {i+1: ch["text"] for i, ch in enumerate(page_chunks)}
    total_pages = len(page_chunks)

    # Step 2: 마크다운 → 트리 (Stack 기반)
    root = TreeNode(
        id="root", title=pdf_path.name, level=0,
        page_start=1, page_end=total_pages, content="",
        heading_type="root",
    )
    stack = [(0, root)]                    # (level, node)
    buffer = []                             # 현재 노드에 누적될 텍스트

    def flush_content():
        """누적된 텍스트를 현재 노드의 content에 저장"""
        if buffer and stack:
            content = "\n".join(buffer).strip()
            if content:
                stack[-1][1].content += "\n\n" + content
                if not stack[-1][1].summary:
                    stack[-1][1].summary = content.replace("#", "").strip()[:300]
        buffer.clear()

    def estimate_page(line_idx, total_lines):
        """줄 위치 기반 page 추정"""
        if total_lines == 0:
            return 1
        return min(int(line_idx / total_lines * total_pages) + 1, total_pages)

    lines = full_md.split("\n")
    for idx, line in enumerate(lines):
        stripped = line.strip()
        if stripped.startswith("#"):
            flush_content()
            # '#' 개수로 level 결정
            level = len(stripped.split()[0]) if stripped.split() else 0
            title = stripped.lstrip("#").strip()
            page = estimate_page(idx, len(lines))
            slug = "_".join(re.findall(r"\w+", title))[:20]
            new_node = TreeNode(
                id=f"{slug}_{idx}", title=title, level=level,
                page_start=page, page_end=page, content="",
                heading_type=classify_heading(title),
            )
            # 같거나 더 얕은 level은 pop (부모 찾기)
            while stack and stack[-1][0] >= level:
                stack.pop()
            # 부모에 attach
            if stack:
                stack[-1][1].children.append(new_node)
                stack[-1][1].page_end = max(stack[-1][1].page_end, page)
            stack.append((level, new_node))
        else:
            buffer.append(line)
    flush_content()

    # Step 3: page_chunks로 page range 보정
    def refine_pages(node: TreeNode):
        if node.content:
            snippet = node.content[:100].strip()
            for pn, ptext in page_contents.items():
                if snippet and snippet in ptext:
                    node.page_start = node.page_end = pn
                    break
        for c in node.children:
            refine_pages(c)
        if node.children:
            # 자식 page range로 부모 page range 갱신
            node.page_start = min(c.page_start for c in node.children + [node])
            node.page_end   = max(c.page_end   for c in node.children + [node])

    refine_pages(root)

    elapsed = time.time() - t0
    n_nodes = 1 + sum(1 for _ in iter_tree(root))
    print(f"Parsed in {elapsed:.2f}s: {total_pages} pages, {n_nodes} nodes")
    return DocumentTree(pdf_path.stem, root, total_pages, str(pdf_path))


def iter_tree(node: TreeNode):
    """노드 순회 generator (root 제외)"""
    for c in node.children:
        yield c
        yield from iter_tree(c)


# Bigtable 논문 파싱 실행
tree = parse_pdf_to_tree(PDF_PATH)
tree.print_tree()

Parsing bigtable-osdi06.pdf with PyMuPDF4LLM ...
=== Document parser messages ===
                                                                                                                                            Using RapidOCR for OCR processing.

=== Document parser messages ===
                                                                                                                                                                               Using RapidOCR for OCR processing.

Parsed in 5.25s: 14 pages, 37 nodes

 bigtable-osdi06 (14 pages)
📑 [L0] bigtable-osdi06.pdf (p1-14)
  📖 [L1] **Bigtable: A Distributed Storage System for Structured Data (p1-13)
    📄 [L2] **Abstract** (p1-1)
    📄 [L2] **1 Introduction** (p1-1)
    📄 [L2] **2 Data Model** (p1-1)
    📄 [L2] **Rows** (p2-2)
    📄 [L2] **Column Families** (p2-2)
    📄 [L2] **Timestamps** (p2-2)
    📄 [L2] Figure 2: Writing to Bigtable. (p3-3)
    📄 [L2] **3 API** (p3-3)
    📄 [L2] Figure 3: Reading from Bigtable

### 4-1-4. 트리 저장 + 레벨별 통계

In [20]:
# ─────────────────────────────────────────────────────
# §4-1-4: 트리 저장 + 통계
# ─────────────────────────────────────────────────────
tree_json = RES / "document_tree.json"
with open(tree_json, "w") as f:
    json.dump(asdict(tree), f, indent=2, default=str)
print(f"트리 저장: {tree_json} ({tree_json.stat().st_size:,} bytes)")

# 레벨별 노드 통계
levels = {}
for n in iter_tree(tree.root):
    levels[n.level] = levels.get(n.level, 0) + 1
print(f"\n레벨별 노드 수: {levels}")

트리 저장: work/results/document_tree.json (94,325 bytes)

레벨별 노드 수: {1: 1, 2: 35}


---
## §4-2 LangGraph Agentic Traversal

`StateGraph`로 `analyze → descend → retrieve → generate` 4-노드 에이전트를 구성한다. 각 노드는 LLM 호출 또는 상태 변환을 수행한다.

이 4-노드 패턴이 본 책에서 가장 자주 재사용되며, Chapter 5의 Three-Stage Architecture에서 production-grade로 확장된다.

### 4-2-1. RetrievalState — 상태 정의 (Annotated reducer)

LangGraph의 핵심 패턴은 `Annotated[..., operator.add]`다. 일반 키는 새 값으로 **덮어쓰지만**, `operator.add` reducer가 붙은 키는 새 값을 **누적**한다.

In [21]:
# ─────────────────────────────────────────────────────
# §4-2-1: RetrievalState 정의
# ─────────────────────────────────────────────────────
class RetrievalState(TypedDict):
    """LangGraph 노드 간 공유되는 상태"""
    query: str
    current_node: Optional[TreeNode]
    tree: TreeNode
    # ↓ 누적되는 필드 (operator.add reducer)
    path_taken:        Annotated[List[str], operator.add]
    retrieved_content: Annotated[List[str], operator.add]
    call_log:          Annotated[List[dict], operator.add]
    # ↓ 덮어쓰는 필드 (default reducer)
    reasoning: str
    confidence: float
    should_descend: bool
    target_child_id: Optional[str]
    depth: int
    final_answer: Optional[str]

print("RetrievalState 정의 완료")

RetrievalState 정의 완료


### 4-2-2. LLM 호출 헬퍼 + JSON 파싱

각 navigate 호출의 입력·출력·지연을 자동 로깅한다. 응답은 JSON이어야 하므로 ``` 펜스를 제거하는 헬퍼도 정의한다.

In [22]:
# ─────────────────────────────────────────────────────
# §4-2-2: LLM 호출 + JSON 파싱 헬퍼
# ─────────────────────────────────────────────────────
def call_llm(prompt: str, call_type: str, call_num: int) -> tuple[str, float]:
    """LLM 호출 + 자동 로깅"""
    print(f"\n  LLM Call #{call_num}  [{call_type.upper()}]")
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,            # 재현 가능성 위해 0
    )
    elapsed = time.time() - t0
    raw = resp.choices[0].message.content.strip()
    print(f"  Latency: {elapsed:.2f}s | Tokens: "
          f"{resp.usage.prompt_tokens} in / {resp.usage.completion_tokens} out")
    return raw, elapsed


def strip_fences(text: str) -> str:
    """LLM이 ```json ...``` 으로 감쌌을 때 JSON만 추출"""
    if "```" in text:
        for part in text.split("```"):
            part = part.strip().lstrip("json").strip()
            try:
                json.loads(part)
                return part
            except Exception:
                continue
    return text

### 4-2-3. 4-노드 정의 — analyze / descend / retrieve / generate

각 노드는 RetrievalState를 받아 일부를 갱신해서 반환하는 함수다. analyze가 LLM 추론을, descend가 트리 이동을, retrieve가 텍스트 추출을, generate가 최종 답을 담당한다.

In [23]:
# ─────────────────────────────────────────────────────
# §4-2-3: 4-노드 정의
# ─────────────────────────────────────────────────────
def analyze_node(state: RetrievalState) -> dict:
    """현재 노드를 LLM에게 보여주고 라우팅 결정을 받는다"""
    if state.get("current_node") is None:
        # 첫 진입 — tree로부터 root 추출
        tree_obj = state["tree"]
        node = tree_obj.root if hasattr(tree_obj, "root") else tree_obj
    else:
        node = state["current_node"]
    call_num = len(state["call_log"]) + 1
    depth = state["depth"]

    # 자식 노드들의 메타 정보만 LLM에 노출 (raw content는 제외)
    children_info = [
        {"id": c.id, "title": c.title, "summary": c.summary[:150]}
        for c in node.children
    ]
    prompt = f"""You are navigating a research paper tree to answer a query.

Query: "{state['query']}"

Current node:
  id      : {node.id}
  title   : {node.title}
  summary : {node.summary[:300]}
  pages   : {node.page_start}-{node.page_end}
  preview : {node.content[:500] if node.content else 'N/A'}

Children: {json.dumps(children_info, indent=2) if children_info else "None (leaf)"}

Decide:
  1. confidence       (0-1)
  2. should_descend   (true only if a specific child is more relevant)
  3. target_child_id  (the id of the best child, null if descending=false)
  4. reasoning        (one sentence)

Respond ONLY as valid JSON. No markdown fences."""
    raw, elapsed = call_llm(prompt, "navigate", call_num)
    raw = strip_fences(raw)
    try:
        decision = json.loads(raw)
    except json.JSONDecodeError:
        # 파싱 실패 시 fallback (자식 있으면 첫 자식으로 내려가기)
        decision = {"confidence":0.5, "should_descend":bool(node.children),
                    "target_child_id": (node.children[0].id if node.children else None),
                    "reasoning":"Fallback"}

    # 결정 출력
    print(f"  Decision: {'↓ descend' if decision['should_descend'] else '→ retrieve'} "
          f"(conf {decision['confidence']:.0%})")
    print(f"  Reasoning: {decision['reasoning']}")

    return {
        "path_taken":      [node.id],
        "current_node":    node,
        "confidence":      float(decision["confidence"]),
        "should_descend":  bool(decision["should_descend"]),
        "target_child_id": decision.get("target_child_id"),
        "reasoning":       decision["reasoning"],
        "depth":           depth + 1,
        "call_log":        [{"call_number":call_num, "call_type":"navigate",
                             "node_id":node.id, "depth":depth,
                             "confidence":decision["confidence"],
                             "latency_s":round(elapsed,3)}],
    }


def descend(state: RetrievalState) -> dict:
    """current_node를 target_child로 이동"""
    current = state["current_node"]
    target_id = state.get("target_child_id")
    target = next((c for c in current.children if c.id == target_id),
                  current.children[0])
    print(f"\n  ➜ Descending → \"{target.title}\"")
    return {"current_node": target}


def retrieve_content(state: RetrievalState) -> dict:
    """현재 노드의 content를 retrieved_content에 추가"""
    node = state["current_node"]
    print(f"\n  ✦ Retrieving content from: \"{node.title}\"")
    print(f"     pages {node.page_start}-{node.page_end} | {len(node.content)} chars")
    chunk = f"=== {node.title} (Pages {node.page_start}-{node.page_end}) ===\n{node.content}"
    return {"retrieved_content": [chunk]}


def generate_answer(state: RetrievalState) -> dict:
    """retrieved_content로 최종 답변 생성 (마지막 LLM 호출)"""
    call_num = len(state["call_log"]) + 1
    context = "\n\n---\n\n".join(state["retrieved_content"])
    prompt = f"""You are an expert. Answer the question using ONLY the retrieved sections.
Cite section title and page range. If insufficient, say so.

Question: {state['query']}

Retrieved sections:
{context}

Answer:"""
    raw, elapsed = call_llm(prompt, "answer", call_num)
    return {
        "final_answer": raw,
        "call_log":     [{"call_number":call_num, "call_type":"answer",
                          "latency_s":round(elapsed,3)}],
    }

print("4개 그래프 노드 정의 완료")

4개 그래프 노드 정의 완료


### 4-2-4. 라우팅 정책 + 그래프 조립

3개 종료 조건: (1) confidence < 0.3, (2) MAX_DEPTH 도달, (3) frontier 비어있음. 그렇지 않으면 should_descend에 따라 다음 노드 결정.

In [24]:
# ─────────────────────────────────────────────────────
# §4-2-4: 라우팅 + 그래프 조립
# ─────────────────────────────────────────────────────
from langgraph.graph import StateGraph, END
from typing import Literal

MAX_DEPTH = 5

def route(state: RetrievalState) -> Literal["descend", "retrieve", "end"]:
    """4-노드 그래프의 라우팅 정책"""
    if state["confidence"] < 0.3:
        print(f"\n  ✗ Low confidence ({state['confidence']:.0%}) — END")
        return "end"
    if state["depth"] >= MAX_DEPTH:
        print(f"\n  ⚠ Max depth ({MAX_DEPTH}) reached → retrieve")
        return "retrieve"
    if state["should_descend"] and state["current_node"].children:
        return "descend"
    return "retrieve"

# StateGraph 조립
workflow = StateGraph(RetrievalState)
workflow.add_node("analyze",  analyze_node)
workflow.add_node("descend",  descend)
workflow.add_node("retrieve", retrieve_content)
workflow.add_node("generate", generate_answer)

workflow.set_entry_point("analyze")
workflow.add_conditional_edges("analyze", route,
    {"descend":"descend", "retrieve":"retrieve", "end":END})
workflow.add_edge("descend",  "analyze")  # ← 루프!
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

graph = workflow.compile()
print("LangGraph 컴파일 완료")

LangGraph 컴파일 완료


---
## §4-3 Bigtable 논문 케이스 스터디

직접 빌드한 시스템으로 'What is Bigtable and what problem does it solve?'를 질의한다. 트래버설 경로와 LLM 호출이 모두 출력된다.

In [25]:
# ─────────────────────────────────────────────────────
# §4-3: Bigtable 논문 질의 실행
# ─────────────────────────────────────────────────────
query = "What is Bigtable and what problem does it solve?"

t_start = time.time()
result = graph.invoke({
    "query": query,
    "current_node": None,
    "tree": tree.root,            # DocumentTree.root 직접 전달
    "path_taken": [],
    "retrieved_content": [],
    "reasoning": "",
    "confidence": 0.0,
    "should_descend": True,
    "target_child_id": None,
    "depth": 0,
    "final_answer": None,
    "call_log": [],
})
total_s = time.time() - t_start

print("\n" + "="*60)
print("  RETRIEVAL SUMMARY")
print("="*60)
print(f"  Total LLM calls : {len(result['call_log'])}")
print(f"  Path taken      : {' → '.join(result['path_taken'])}")
print(f"  Total latency   : {total_s:.2f}s")
print(f"  Confidence      : {result['confidence']:.0%}")
print("\n  [Answer]")
print(result["final_answer"])


  LLM Call #1  [NAVIGATE]
  Latency: 1.61s | Tokens: 242 in / 66 out
  Decision: ↓ descend (conf 99%)
  Reasoning: The child is the paper titled "Bigtable: A Distributed Storage System for Structured Data," which directly matches the query asking what Bigtable is and what problem it solves.

  ➜ Descending → "**Bigtable: A Distributed Storage System for Structured Data**"

  LLM Call #2  [NAVIGATE]
  Latency: 0.92s | Tokens: 2413 in / 59 out
  Decision: ↓ descend (conf 99%)
  Reasoning: The Abstract directly states what Bigtable is and summarizes the problem it solves, making it the most relevant child for this query.

  ➜ Descending → "**Abstract**"

  LLM Call #3  [NAVIGATE]
  Latency: 1.32s | Tokens: 296 in / 70 out
  Decision: → retrieve (conf 99%)
  Reasoning: The abstract already directly answers the query by defining Bigtable as a distributed storage system for structured data that scales to petabytes across thousands of commodity servers, which addresses the problem of storing

In [26]:
# ─────────────────────────────────────────────────────
# 호출 로그를 표로 정리
# ─────────────────────────────────────────────────────
import pandas as pd
df_log = pd.DataFrame(result["call_log"])
df_log

,call_number,call_type,node_id,depth,confidence,latency_s
0,1,navigate,root,0.0,0.99,1.612
1,2,navigate,Bigtable_A_Distribut_0,1.0,0.99,0.921
2,3,navigate,Abstract_8,2.0,0.99,1.319
3,4,answer,NaN,NaN,NaN,1.278


---
## §4-4 관찰 가능성과 튜닝

VectorlessRAG의 가장 큰 장점은 모든 LLM 호출의 입력·출력·결정·신뢰도가 자연어로 기록된다는 점이다. 임베딩 RAG는 '왜 이 청크가 뽑혔는가'를 설명하기 어렵지만, 본 시스템은 그렇지 않다.

이 trace 패턴이 Chapter 5의 production Verifier 패턴의 직접 기반이다.

### 4-4-1. MAX_DEPTH 변경 실험

MAX_DEPTH를 줄이면 트래버설이 일찍 끝나 비용은 줄지만 deep node에 답이 있을 때 정확도가 떨어질 수 있다.

In [27]:
# ─────────────────────────────────────────────────────
# §4-4-1: MAX_DEPTH 변경 실험
# ─────────────────────────────────────────────────────
def run_query(query: str, max_depth: int = 5):
    """MAX_DEPTH를 임의로 변경하여 동일 쿼리 실행"""
    global MAX_DEPTH
    MAX_DEPTH = max_depth
    t0 = time.time()
    out = graph.invoke({
        "query": query, "current_node": None, "tree": tree.root,
        "path_taken": [], "retrieved_content": [], "reasoning": "",
        "confidence": 0.0, "should_descend": True, "target_child_id": None,
        "depth": 0, "final_answer": None, "call_log": [],
    })
    return {
        "max_depth": max_depth,
        "total_calls": len(out["call_log"]),
        "path_length": len(out["path_taken"]),
        "latency_s": round(time.time() - t0, 2),
        "answer_preview": out["final_answer"][:150],
    }

# Bigtable 1개 질의로 두 설정 비교
print("MAX_DEPTH=3:")
print(run_query("What is Bigtable?", max_depth=3))
print("\nMAX_DEPTH=5:")
print(run_query("What is Bigtable?", max_depth=5))

MAX_DEPTH=3:

  LLM Call #1  [NAVIGATE]
  Latency: 0.95s | Tokens: 236 in / 64 out
  Decision: ↓ descend (conf 99%)
  Reasoning: The child is the paper titled "Bigtable: A Distributed Storage System for Structured Data," which directly answers what Bigtable is.

  ➜ Descending → "**Bigtable: A Distributed Storage System for Structured Data**"

  LLM Call #2  [NAVIGATE]
  Latency: 0.85s | Tokens: 2407 in / 47 out
  Decision: ↓ descend (conf 99%)
  Reasoning: The Abstract child directly defines Bigtable and is the most relevant place to answer what it is.

  ➜ Descending → "**Abstract**"

  LLM Call #3  [NAVIGATE]
  Latency: 0.84s | Tokens: 290 in / 50 out
  Decision: → retrieve (conf 99%)
  Reasoning: The abstract directly defines Bigtable as a distributed storage system for structured data at massive scale, which answers the query fully.

  ⚠ Max depth (3) reached → retrieve

  ✦ Retrieving content from: "**Abstract**"
     pages 1-1 | 830 chars

  LLM Call #4  [ANSWER]
  Latency: 0.98

### 4-4-2. 쿼리당 비용 추정

GPT-4o-mini 단가로 1회 질의의 비용을 추정한다. 같은 정확도 기준 임베딩 RAG의 약 5~20배 비싸지만, 추적성·인용 정확도가 압도적이다.

In [28]:
# ─────────────────────────────────────────────────────
# §4-4-2: 비용 추정 (gpt-4o-mini 기준)
# ─────────────────────────────────────────────────────
COST_INPUT  = 0.15 / 1_000_000   # $0.15 / 1M tok
COST_OUTPUT = 0.60 / 1_000_000   # $0.60 / 1M tok

# 호출별 토큰은 result.call_log에 기록되지 않으므로 대략치만
total_calls = len(result["call_log"])
approx_cost = total_calls * (1500 * COST_INPUT + 200 * COST_OUTPUT)

print(f"\n질의 1회당 추정 비용: ~${approx_cost:.4f}")
print(f"같은 질의 1000회: ~${approx_cost*1000:.2f}")
print()
print("→ VectorlessRAG (직접 구현)는 임베딩 RAG 대비 비용이 약 5~20배 비싸지만,")
print("  답변의 추적성·인용 정확도는 압도적으로 높다.")
print("→ Chapter 5는 이 비용 위에 Verifier·refusal을 추가하여")
print("  hallucination을 결정적으로 차단한다.")


질의 1회당 추정 비용: ~$0.0014
같은 질의 1000회: ~$1.38

→ VectorlessRAG (직접 구현)는 임베딩 RAG 대비 비용이 약 5~20배 비싸지만,
  답변의 추적성·인용 정확도는 압도적으로 높다.
→ Chapter 5는 이 비용 위에 Verifier·refusal을 추가하여
  hallucination을 결정적으로 차단한다.


### 4-4-3. Ch.3 PageIndex vs 본 챕터 직접 구현 비교

In [29]:
# ─────────────────────────────────────────────────────
# §4-4-3: Ch.3 PageIndex vs Ch.4 직접 구현 비교표
# ─────────────────────────────────────────────────────
comparison = pd.DataFrame([
    {"축": "코드량",          "Ch.3 (PageIndex 개념)": "~30줄",   "Ch.4 (직접 구현)": "~300줄"},
    {"축": "트리 빌더",       "Ch.3 (PageIndex 개념)": "managed", "Ch.4 (직접 구현)": "stack-based 30줄"},
    {"축": "라우팅 정책",     "Ch.3 (PageIndex 개념)": "내장",    "Ch.4 (직접 구현)": "MAX_DEPTH·conf 임의"},
    {"축": "로깅",            "Ch.3 (PageIndex 개념)": "API 응답", "Ch.4 (직접 구현)": "전체 호출 log 기록"},
    {"축": "비용 (Bigtable)", "Ch.3 (PageIndex 개념)": "~$0.001",  "Ch.4 (직접 구현)": "~$0.001"},
    {"축": "커스터마이즈",    "Ch.3 (PageIndex 개념)": "제한적",   "Ch.4 (직접 구현)": "자유"},
    {"축": "권장 시나리오",   "Ch.3 (PageIndex 개념)": "MVP",      "Ch.4 (직접 구현)": "도메인 특화"},
])
comparison

,축,Ch.3 (PageIndex 개념),Ch.4 (직접 구현)
0,코드량,~30줄,~300줄
1,트리 빌더,managed,stack-based 30줄
2,라우팅 정책,내장,MAX_DEPTH·conf 임의
3,로깅,API 응답,전체 호출 log 기록
4,비용 (Bigtable),~$0.001,~$0.001
5,커스터마이즈,제한적,자유
6,권장 시나리오,MVP,도메인 특화
